In [ ]:
import json
import data_utils
import conceptset_utils

In [ ]:
"""
CLASS_SIM_CUTOFF: Concenpts with cos similarity higher than this to any class will be removed
OTHER_SIM_CUTOFF: Concenpts with cos similarity higher than this to another concept will be removed
MAX_LEN: max number of characters in a concept

PRINT_PROB: what percentage of filtered concepts will be printed
"""

CLASS_SIM_CUTOFF = 0.85
OTHER_SIM_CUTOFF = 0.9
MAX_LEN = 30
PRINT_PROB = 1

dataset = "imagenet"
device = "cuda"

save_name = "data/concept_sets/{}_filtered_byclass.json".format(dataset)

In [ ]:
#EDIT these to use the initial concept sets you want

with open("data/concept_sets/gpt3_init/gpt3_{}_important.json".format(dataset), "r") as f:
    important_dict = json.load(f)
with open("data/concept_sets/gpt3_init/gpt3_{}_superclass.json".format(dataset), "r") as f:
    superclass_dict = json.load(f)
with open("data/concept_sets/gpt3_init/gpt3_{}_around.json".format(dataset), "r") as f:
    around_dict = json.load(f)
    
with open(data_utils.LABEL_FILES[dataset], "r") as f:
    classes = f.read().split("\n")

In [ ]:
concepts = set()

for values in important_dict.values():
    concepts.update(set(values))

for values in superclass_dict.values():
    concepts.update(set(values))
    
for values in around_dict.values():
    concepts.update(set(values))

print(len(concepts))

In [ ]:
concepts = conceptset_utils.remove_too_long(concepts, MAX_LEN, PRINT_PROB)

In [ ]:
concepts = conceptset_utils.filter_too_similar_to_cls(concepts, classes, CLASS_SIM_CUTOFF, device, PRINT_PROB)

In [ ]:
concepts = conceptset_utils.filter_too_similar(concepts, OTHER_SIM_CUTOFF, device, PRINT_PROB)

In [ ]:
concept_by_class = {}
for label in classes:
    concept_by_class[label] = set()
    for values in important_dict[label]:
        if values in concepts:
            concept_by_class[label].add(values)

    for values in superclass_dict[label]:
        if values in concepts:
            concept_by_class[label].add(values)

        
    for values in around_dict[label]:
        if values in concepts:
            concept_by_class[label].add(values)

    concept_by_class[label] = sorted(list(concept_by_class[label]))


In [ ]:
json_object = json.dumps(concept_by_class, indent=4)
with open(save_name, "w") as f:
    f.write(json_object)